# Medical LLM Notebook 01 — Dataset Preparation

Prepare the reproducible MedHallu methodological-validation cohort used in the manuscript.

This notebook:
- loads `UTAustin-AIHealth/MedHallu`, configuration `pqa_labeled`;
- verifies the 1,000-row working split;
- performs difficulty-stratified sampling of 100 matched question pairs with random seed 42;
- assigns `Pair_ID`;
- exports both wide (100-pair) and long (200-response) formats;
- saves pair identifiers and metadata for repository reproducibility.

The original MedHallu dataset should be obtained from its official distribution source rather than redistributed in this repository.

In [ ]:
!pip -q install datasets scikit-learn pandas numpy

In [ ]:
import os
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

SEED = 42
DATASET_ID = "UTAustin-AIHealth/MedHallu"
CONFIG = "pqa_labeled"
SPLIT = "train"

print("Seed:", SEED)


## 1. Load the official MedHallu working split

In [ ]:
ds = load_dataset(
    DATASET_ID,
    CONFIG,
    split=SPLIT
)

df = ds.to_pandas()

print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())


In [ ]:
required = [
    "Question",
    "Knowledge",
    "Ground Truth",
    "Difficulty Level",
    "Hallucinated Answer",
    "Category of Hallucination"
]

missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing expected columns: {missing}"

assert len(df) == 1000, (
    f"Expected the 1,000-row pqa_labeled working split, found {len(df)}."
)

print("\nDifficulty distribution:")
display(df["Difficulty Level"].value_counts())

print("\nHallucination category distribution:")
display(df["Category of Hallucination"].value_counts())

print("\nMissing values:")
display(df[required].isna().sum())


## 2. Difficulty-stratified 100-pair sample

In [ ]:
_, pilot = train_test_split(
    df,
    test_size=100,
    random_state=SEED,
    stratify=df["Difficulty Level"]
)

pilot = pilot.reset_index(drop=True).copy()

pilot.insert(
    0,
    "Pair_ID",
    [f"MEDH_{i:03d}" for i in range(1, len(pilot) + 1)]
)

pairs_df = pilot.rename(
    columns={
        "Ground Truth": "Ground_Truth",
        "Difficulty Level": "Difficulty",
        "Hallucinated Answer": "Hallucinated_Answer",
        "Category of Hallucination": "Hallucination_Category"
    }
)[
    [
        "Pair_ID",
        "Question",
        "Knowledge",
        "Ground_Truth",
        "Hallucinated_Answer",
        "Difficulty",
        "Hallucination_Category"
    ]
].copy()

print("Selected pairs:", len(pairs_df))
display(pairs_df["Difficulty"].value_counts())
display(pairs_df["Hallucination_Category"].value_counts())
display(pairs_df.head())


## 3. Create 200-response long format

In [ ]:
factual = pairs_df[
    [
        "Pair_ID",
        "Question",
        "Knowledge",
        "Ground_Truth",
        "Difficulty",
        "Hallucination_Category"
    ]
].copy()

factual = factual.rename(
    columns={"Ground_Truth": "Response"}
)
factual["Response_Type"] = "Factual"
factual["Label"] = 0

hall = pairs_df[
    [
        "Pair_ID",
        "Question",
        "Knowledge",
        "Hallucinated_Answer",
        "Difficulty",
        "Hallucination_Category"
    ]
].copy()

hall = hall.rename(
    columns={"Hallucinated_Answer": "Response"}
)
hall["Response_Type"] = "Hallucinated"
hall["Label"] = 1

long_df = pd.concat(
    [factual, hall],
    ignore_index=True
).sort_values(
    ["Pair_ID", "Label"]
).reset_index(drop=True)

assert len(long_df) == 200
assert long_df["Pair_ID"].nunique() == 100
assert (long_df.groupby("Pair_ID").size() == 2).all()

print(long_df["Response_Type"].value_counts())
display(long_df.head(6))


## 4. Export reproducibility files

In [ ]:
OUT = "/content/Notebook01_outputs"
os.makedirs(OUT, exist_ok=True)

pairs_df.to_csv(
    f"{OUT}/MedHallu_Pilot_100_Pairs.csv",
    index=False
)

long_df.to_csv(
    f"{OUT}/MedHallu_Pilot_200.csv",
    index=False
)

pairs_df[["Pair_ID"]].to_csv(
    f"{OUT}/selected_pair_ids.csv",
    index=False
)

pairs_df[
    ["Pair_ID", "Difficulty", "Hallucination_Category"]
].to_csv(
    f"{OUT}/sample_metadata.csv",
    index=False
)

print("Saved:")
for f in sorted(os.listdir(OUT)):
    print(" -", f)


In [ ]:
from google.colab import files
import shutil

archive = shutil.make_archive(
    "/content/Notebook01_Dataset_Preparation_Outputs",
    "zip",
    OUT
)

files.download(archive)
